### RapidFire AI RAG/Context Engineering: RapidFire AI Documents Q&A

In [2]:
# import getpass
# TRITON_API_KEY = getpass.getpass("Enter your Triton API key: ")

# The above option is probably safer
from pathlib import Path
TRITON_API_KEY = Path("~/Project1/api-key.txt").expanduser().read_text(encoding="utf-8").splitlines()[0].strip()

In [3]:
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)

from rapidfireai import Experiment

import re, json, math
from typing import List as listtype, Dict, Any

# project1_eval.py must be in the same directory as this notebook.
# It is shared across evaluate_retrieval.py, run_judge.py, and this notebook;
# do not modify the file itself.
from eval_scripts.project1_eval import (
    to_spans,           # validates + coerces (file, start, end) tuples
    precision_at_k,     # overlap-aware Precision@k
    recall_at_k,        # overlap-aware Recall@k
    f1_at_k,            # harmonic mean of the above
    call_judge,         # LLM-as-judge: correctness / faithfulness / completeness
)


KeyboardInterrupt: 

In [ ]:
import os
os.environ["RF_MLFLOW_ENABLED"] = "true"

In [ ]:
# Load TensorBoard extension
# %load_ext tensorboard

# TensorBoard log directory will be auto-created in experiment path

##### ⚠️ API Cost Considerations
This notebook runs 4 configurations concurrently on a downsampled dataset of 256 examples.
Estimated Costs:
- Current run (downsampled): \$5 
- Full set: \$45

> 💡 **Tip:** Monitor your API usage to avoid unexpected charges.

### Load Queries to `qrels`

In [ ]:
import pandas as pd
from datasets import Dataset
from pathlib import Path

# Load golden QA pairs from validation set
validation_file = Path("~/Project1/validation-set-golden-qa-pairs.json").expanduser()
with open(validation_file, "r") as f:
    qa_pairs = json.load(f)

# Build queries dataset.
# reference_answer is passed through the batch so postprocess_fn can read it
# without reloading the file on every batch.
queries = {
    "query":            [qa["question"]         for qa in qa_pairs],
    "query_id":         [str(qa["question_id"]) for qa in qa_pairs],
    "reference_answer": [qa["reference_answer"] for qa in qa_pairs],
}
queries = Dataset.from_dict(queries).shuffle(seed=42)

# Build qrels: one row per (query_id, evidence span).
# lines is normalised to a strict 2-element [start_line, end_line] list so it
# is always compatible with project1_eval.to_spans({file, lines: [s, e]}).
qrels_data = []
for qa in qa_pairs:
    query_id = str(qa["question_id"])
    for evidence in qa["source_evidence"]:
        raw_lines = evidence.get("lines", [])
        if len(raw_lines) >= 2:
            start_line, end_line = int(min(raw_lines)), int(max(raw_lines))
        elif len(raw_lines) == 1:
            start_line = end_line = int(raw_lines[0])
        else:
            # No line info — skip; a span with unknown bounds cannot be evaluated
            continue
        qrels_data.append({
            "query_id":   query_id,
            "file":       Path(evidence.get("file", "")).name,
            "start_line": start_line,
            "end_line":   end_line,
            "relevance":  1,
        })

qrels = pd.DataFrame(qrels_data)
print(f"Loaded {len(qa_pairs)} QA pairs")
print(f"Dataset shape: {queries.shape}")
print(f"Qrels shape:   {qrels.shape}")
qrels.head(10)


Loaded 45 QA pairs
Dataset shape: (45, 3)
Qrels shape:   (52, 5)


,query_id,file,start_line,end_line,relevance
0,1,configs.rst,23,48,1
1,2,difference.rst,4,29,1
2,3,experiment.rst,27,39,1
3,4,configs.rst,58,102,1
4,5,walkthroughrag.rst,29,103,1
5,6,icops.rst,75,176,1
6,7,evalsfunctions.rst,8,31,1
7,8,ragspecs.rst,56,98,1
8,8,rag_fiqa.rst,49,59,1
9,9,experiment.rst,215,219,1


### Create Experiment

In [ ]:
# from rapidfireai import Experiment
my_experiment = "exp_0501_1"
experiment = Experiment(experiment_name=my_experiment, mode="evals")

An experiment with the same name already exists. Created a new experiment 'exp_0501_7' with Experiment ID: 32 at /home/yup031/rapidfireai/rapidfire_experiments/exp_0501_7


### Define Partial Multi-Config Knobs for LangChain part of RAG Pipeline using RapidFire AI Wrapper APIs

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Chunking utilities
#
# Two splitter wrappers are provided for comparison in the grid search:
#
#   LineNumberSplitter      – wraps RecursiveCharacterTextSplitter.
#                             Passes add_start_index=True to the inner splitter
#                             so char offsets are available, then tags each chunk
#                             with span_coord="char" so compute_metrics_fn knows
#                             to convert char offsets → line numbers at eval time.
#
#   StructuralRSTSplitter   – wraps rst_structural_splitter.
#                             Splits on RST structural boundaries and writes
#                             start_line/end_line natively. Tags each chunk with
#                             span_coord="line" so no conversion is needed.
#
# Metadata written per chunk (both strategies)
# --------------------------------------------
#   source      str  – absolute path (set by TextLoader)
#   span_coord  str  – "char" | "line"  (consumed by compute_metrics_fn)
#
# Additionally per strategy:
#   LineNumberSplitter    → start_index, end_index  (char offsets)
#   StructuralRSTSplitter → start_line, end_line    (1-based line numbers)
# ─────────────────────────────────────────────────────────────────────────────

import re
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import TextSplitter


# ── rst_structural_splitter ──────────────────────────────────────────────────

_RST_HEADING_RE = re.compile(r"^([=\-~^'`#*+])(\1*)$")

def rst_structural_splitter(
    documents: list[Document],
    min_tokens: int = 64,
    max_tokens: int = 256,
    tokenizer_encoding: str = "gpt2",
) -> list[Document]:
    """Split RST documents on structural boundaries (sections, paragraphs).

    Writes start_line / end_line (1-based, inclusive) directly into metadata.
    Falls back to sentence splitting for paragraphs that exceed max_tokens.
    """
    import tiktoken
    enc = tiktoken.get_encoding(tokenizer_encoding)

    def token_len(text: str) -> int:
        return len(enc.encode(text))

    chunks: list[Document] = []

    for doc in documents:
        src   = doc.metadata.get("source", "")
        lines = doc.page_content.splitlines(keepends=True)

        # pass 1: locate section boundaries
        section_starts = [0]
        for i, line in enumerate(lines):
            stripped = line.rstrip("\n")
            if _RST_HEADING_RE.match(stripped) and len(stripped) >= 2:
                title_idx = max(0, i - 1)
                if title_idx not in section_starts:
                    section_starts.append(title_idx)

        section_starts = sorted(set(section_starts))
        section_ranges = list(zip(section_starts, section_starts[1:] + [len(lines)]))

        for sec_start, sec_end in section_ranges:
            section_lines = lines[sec_start:sec_end]

            # pass 2: split into paragraphs on blank lines
            paragraphs: list[list[tuple[int, str]]] = []
            current:    list[tuple[int, str]]        = []
            for offset, line in enumerate(section_lines):
                lineno = sec_start + offset + 1  # 1-based
                if line == "" and current:
                    paragraphs.append(current)
                    current = []
                else:
                    current.append((lineno, line))
            if current:
                paragraphs.append(current)

            # pass 3: merge tiny paragraphs; split oversized ones
            # flush is defined outside the section loop to avoid closure capture issues
            def _flush(
                buf: list[tuple[int, str]],
                _src: str,
                chunk_type: str = "paragraph",
            ) -> None:
                if not buf:
                    return
                text = "".join(t for _, t in buf).strip()
                if not text:
                    return
                chunks.append(Document(
                    page_content=text,
                    metadata={
                        "source":     _src,
                        "start_line": buf[0][0],
                        "end_line":   buf[-1][0],
                        "chunk_type": chunk_type,
                        "span_coord": "line",
                    },
                ))

            buffer: list[tuple[int, str]] = []

            for para in paragraphs:
                para_text   = "".join(t for _, t in para).strip()
                para_tokens = token_len(para_text)

                if para_tokens > max_tokens:
                    # Split by individual lines to preserve exact line numbers.
                    # Sentence splitting loses line number info because para_text
                    # is a stripped string with no per-line tracking.
                    _flush(buffer, src)
                    buffer = []
                    line_buf:  list[tuple[int, str]] = []
                    line_toks  = 0
                    for lineno, line_text in para:
                        t = token_len(line_text)
                        if line_toks + t > max_tokens and line_buf:
                            _flush(line_buf, src, chunk_type="overflow")
                            line_buf  = []
                            line_toks = 0
                        line_buf.append((lineno, line_text))
                        line_toks += t
                    _flush(line_buf, src, chunk_type="overflow")

                elif token_len("".join(t for _, t in buffer)) + para_tokens < min_tokens:
                    buffer.extend(para)

                else:
                    _flush(buffer, src)
                    buffer = para[:]

            _flush(buffer, src)

    return chunks


# ── Splitter wrappers ────────────────────────────────────────────────────────

class LineNumberSplitter(TextSplitter):
    """Wraps RecursiveCharacterTextSplitter; tags chunks with span_coord="char".

    Char offsets (start_index / end_index) are left in metadata as-is.
    compute_metrics_fn reads span_coord and converts them to line numbers
    at eval time using the source files.
    """
    def __init__(self, inner_splitter: TextSplitter):
        # chunk_size/overlap required by TextSplitter.__init__; read from inner.
        super().__init__(chunk_size=inner_splitter._chunk_size,
                        chunk_overlap=inner_splitter._chunk_overlap)
        self._inner = inner_splitter

    def split_documents(self, documents: list[Document]) -> list[Document]:
        chunks = self._inner.split_documents(documents)
        for doc in chunks:
            doc.metadata["span_coord"] = "char"
        return chunks

    def split_text(self, text: str) -> list[str]:
        return self._inner.split_text(text)


class StructuralRSTSplitter(TextSplitter):
    """Wraps rst_structural_splitter; chunks already have span_coord="line"."""

    def __init__(self, min_tokens: int = 64, max_tokens: int =256):
        super().__init__(chunk_size=max_tokens, chunk_overlap=0)
        self.min_tokens = min_tokens
        self.max_tokens = max_tokens

    def split_documents(self, documents: list[Document]) -> list[Document]:
        return rst_structural_splitter(
            documents,
            min_tokens=self.min_tokens,
            max_tokens=self.max_tokens,
        )

    def split_text(self, text: str) -> list[str]:
        doc = Document(page_content=text, metadata={"source": ""})
        return [c.page_content for c in rst_structural_splitter(
            [doc], min_tokens=self.min_tokens, max_tokens=self.max_tokens)]


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from typing import Dict

batch_size = 32

# def custom_template(retrieved_docs: listtype[Document]) -> str:
#     """Serialise retrieved chunks into a single context string.
#     Plain text only; no metadata injected so the prompt stays clean.
#     """
#     return "\n\n".join(doc.page_content for doc in retrieved_docs)


# Shared config reused across all RAG variants
_LOADER = DirectoryLoader(
    path="/home/yup031/Project1/sourcedocs/",
    glob="*.rst",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    sample_seed=1337,
)
_EMBEDDING_CFG = List([{
    "class": OpenAIEmbeddings,
    "model": "api-tgpt-embeddings",
    "api_key": TRITON_API_KEY,
    "base_url": "https://tritonai-api.ucsd.edu",
    "check_embedding_ctx_length": False,
}])
_VECTOR_STORE_CFG = {"type": "faiss", "batch_size": batch_size}
_SEARCH_CFG       = List([{"type": "mmr", "k": 20, "fetch_k": 40, "lambda": 0.9}, {"type": "similarity", "k": 20}])
_RERANKER_CFG     = List([
    # {
    # "class": CrossEncoderReranker,
    # "model_name": "cross-encoder/ms-marco-MiniLM-L6-v2",
    # "model_kwargs": {"device": "cpu"},
    # "top_n": 5,
    # },
    {
    "class": CrossEncoderReranker,
    "model_name": "BAAI/bge-reranker-v2-m3",
    "model_kwargs": {"device": "cpu"},
    "top_n": 3,
    },
    {
    "class": CrossEncoderReranker,
    "model_name": "BAAI/bge-reranker-v2-m3",
    "model_kwargs": {"device": "cpu"},
    "top_n": 5,
    }
    ])

# ─────────────────────────────────────────────────────────────────────────────
# Splitter Configuration: (label, splitter_instance) tuples
# Add new strategies here by appending to _SPLITTER_CONFIGS
# ─────────────────────────────────────────────────────────────────────────────

_SPLITTER_CONFIGS = [
    # ── Strategy A: token-based, chunk_size=256 ──────────────────────────────────
    # LineNumberSplitter wraps the inner splitter and tags each chunk with
    # span_coord="char". Char offsets are converted to line numbers in
    # compute_metrics_fn when to_spans is called.
    # (
    #     "token-256",
    #     LineNumberSplitter(
    #         RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    #             encoding_name="gpt2",
    #             chunk_size=256,
    #             chunk_overlap=32,
    #             add_start_index=True,  # populates start_index / end_index
    #         )
    #     ),
    # ),

    # ── Strategy B: token-based, chunk_size=128 ──────────────────────────────────
    (
        "token-128-overlap-32",
        LineNumberSplitter(
            RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                encoding_name="gpt2",
                chunk_size=128,
                chunk_overlap=32,
                add_start_index=True,
            )
        ),
    ),

    (
        "token-128-overlap-64",
        LineNumberSplitter(
            RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                encoding_name="gpt2",
                chunk_size=128,
                chunk_overlap=64,
                add_start_index=True,
            )
        ),
    ),

    # ── Strategy C: structure-aware RST splitter ─────────────────────────────────
    # StructuralRSTSplitter writes start_line/end_line natively and tags chunks
    # with span_coord="line", so no conversion is needed at eval time.
    # (
    #     "rst-structural_128",
    #     StructuralRSTSplitter(min_tokens=64, max_tokens=128),
    # ),
    (
        "rst-structural_256",
        StructuralRSTSplitter(min_tokens=64, max_tokens=256),
    )
]

# Build RAG specs from splitter configurations
all_splitters = [
    (
        RFLangChainRagSpec(
            document_loader=_LOADER,
            text_splitter=splitter,
            embedding_cfg=_EMBEDDING_CFG,
            vector_store_cfg=_VECTOR_STORE_CFG,
            search_cfg=_SEARCH_CFG,
            reranker_cfg=_RERANKER_CFG,
            enable_gpu_search=False,
            # document_template=custom_template,
        ),
        label,
    )
    for label, splitter in _SPLITTER_CONFIGS
]

### Define Data Processing and Postprocessing Functions

In [ ]:
def openai_sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Retrieve context and build generator prompts.

    Adds to batch
    -------------
    retrieved_spans    list[list[tuple[str, int, int, str]]]
        Per query, rank-ordered (file, start, end, span_coord) tuples.
        start/end are char offsets when span_coord="char" (token splitters),
        or 1-based line numbers when span_coord="line" (structural splitter).
        Conversion to line numbers and to_spans validation happen in
        compute_metrics_fn, not here.
    serialized_context list[str]
        The exact text string passed to the generator for each query.
    prompts            list[list[dict]]
        OpenAI-style message list for each query.
    """
    import tiktoken
    from pathlib import Path

    SYSTEM_INSTRUCTIONS = (
        "You are a technical QA assistant for RapidFire AI documentation. "
        "Answer the question using ONLY the provided context. "
        "Your answer will be scored on three criteria: "
        "Correctness (matches the reference answer), "
        "Faithfulness (no information beyond the context), and "
        "Completeness (covers all key points in the context relevant to the question). "
        "You must provide the answer that best meets all three criteria."
    )
    MAX_TOKENS = 2000

    # Batched retrieval → list[list[Document]]
    all_context = rag.get_context(batch_queries=batch["query"], serialize=False)

    # Store raw (file, start, end, span_coord) — no conversion or validation here.
    # span_coord is written by the splitter wrapper (LineNumberSplitter or
    # StructuralRSTSplitter) and tells compute_metrics_fn which coordinate system
    # the start/end values are in.
    retrieved_spans = [
        [
            (
                Path(doc.metadata.get("source", "")).name,
                doc.metadata.get("start_index", doc.metadata.get("start_line", 0)),
                doc.metadata.get("end_index",   doc.metadata.get("end_line",   -1)),
                doc.metadata.get("span_coord", "char"),
            )
            for doc in docs
        ]
        for docs in all_context
    ]

    serialized_context: listtype[str] = rag.serialize_documents(all_context)
    batch["query_id"] = [str(query_id) for query_id in batch["query_id"]]

    enc = tiktoken.get_encoding("gpt2")

    def truncate_to_tokens(text: str, max_tokens: int) -> str:
        tokens = enc.encode(text)
        return enc.decode(tokens[:max_tokens]) if len(tokens) > max_tokens else text

    prompts = []
    for question, context in zip(batch["query"], serialized_context):
        user_content = truncate_to_tokens(
            f"Question: {question}\n\nContext:\n{context}\n\nAnswer:",
            MAX_TOKENS,
        )
        prompts.append([
            {"role": "system", "content": SYSTEM_INSTRUCTIONS},
            {"role": "user",   "content": user_content},
        ])

    return {
        "prompts":            prompts,
        "retrieved_spans":    retrieved_spans,
        "serialized_context": serialized_context,
        **batch,
    }


def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Attach ground-truth fields after generation.

    Adds to batch
    -------------
    ground_truth_spans  list[list[tuple[str, int, int]]]
        Raw (file, start_line, end_line) tuples from qrels.
        to_spans validation happens in compute_metrics_fn.
    """
    ground_truth_spans = []
    for query_id in batch["query_id"]:
        rows = qrels[qrels["query_id"] == str(query_id)]
        # qrels always stores real line numbers (built in cell 6)
        raw = [
            (row["file"], int(row["start_line"]), int(row["end_line"]))
            for _, row in rows.iterrows()
        ]
        ground_truth_spans.append(raw)  # to_spans called in compute_metrics_fn

    batch["ground_truth_spans"] = ground_truth_spans

    assert "reference_answer" in batch, (
        "reference_answer missing from batch — make sure it is included when "
        "building the queries Dataset in cell 6."
    )
    return batch


### Define Custom Eval Metrics Functions

In [ ]:
# All retrieval metrics delegate to project1_eval so scoring is identical to
# the TA's evaluate_retrieval.py and run_judge.py scripts.

JUDGE_MODEL    = "api-llama-4-scout"    #"api-gpt-oss-120b"
JUDGE_BASE_URL = "https://tritonai-api.ucsd.edu"
K = 5

# Module-level cache: source filename → full file text.
# Populated lazily on first access; persists across batches within a session.
_file_line_cache: dict[str, str] = {}


def _char_offsets_to_lines(fname: str, start_char: int, end_char: int) -> tuple[int, int]:
    """Convert char offsets to 1-based inclusive line numbers.

    Looks up the source file under ~/Project1/sourcedocs. Returns (1, 1) if
    the file cannot be found so evaluation degrades gracefully rather than
    crashing.
    """
    if fname not in _file_line_cache:
        candidates = list(Path("~/Project1/sourcedocs").expanduser().rglob(fname))
        if not candidates:
            return 1, 1
        _file_line_cache[fname] = candidates[0].read_text(encoding="utf-8")
    text     = _file_line_cache[fname]
    end_char = end_char if end_char >= 0 else len(text)
    start_line = text[:start_char].count("\n") + 1
    end_line   = text[:end_char].count("\n")   + 1
    return start_line, end_line


def _normalise_retrieved_spans(
    raw_spans: list[tuple[str, int, int, str]],
) -> list[tuple[str, int, int]]:
    """Convert raw retrieved spans to validated (file, start_line, end_line) tuples.

    Reads span_coord (4th element) to decide whether conversion is needed:
      "char" → convert char offsets to line numbers via _char_offsets_to_lines
      "line" → already line numbers; pass through directly
    Then validates and deduplicates through project1_eval.to_spans.
    """
    converted = []
    for fname, start, end, coord in raw_spans:
        if coord == "char":
            start_line, end_line = _char_offsets_to_lines(fname, start, end)
        else:
            start_line, end_line = int(start), int(end)
        converted.append((fname, start_line, end_line))
    return to_spans(converted)


def _normalise_gt_spans(
    raw_spans: list[tuple[str, int, int]],
) -> list[tuple[str, int, int]]:
    """Validate and deduplicate ground-truth spans through project1_eval.to_spans.

    qrels always stores real line numbers so no coordinate conversion needed.
    """
    return to_spans(raw_spans)


def sample_compute_metrics_fn(batch: Dict[str, listtype]) -> Dict[str, Dict[str, Any]]:
    """Compute retrieval and generation metrics for one batch.

    Reads from batch fields:
        retrieved_spans    list[list[tuple[str, int, int, str]]]  (with span_coord)
        ground_truth_spans list[list[tuple[str, int, int]]]       (always line numbers)
        serialized_context list[str]
        generated_text     list[str]
        reference_answer   list[str]
        query              list[str]
    """
    total_queries = len(batch["query"])

    precisions, recalls, f1s = [], [], []
    correctness_scores, faithfulness_scores, completeness_scores = [], [], []
    judge_failures = 0

    for pred_spans_raw, gt_spans_raw, context, gen_text, ref_ans, query in zip(
        batch["retrieved_spans"],
        batch["ground_truth_spans"],
        batch["serialized_context"],
        batch["generated_text"],
        batch["reference_answer"],
        batch["query"],
    ):
        # Convert + validate here, just before scoring
        pred_spans = _normalise_retrieved_spans(pred_spans_raw)
        gt_spans   = _normalise_gt_spans(gt_spans_raw)

        precisions.append(precision_at_k(pred_spans, gt_spans, k=K))
        recalls.append(recall_at_k(pred_spans,       gt_spans, k=K))
        f1s.append(f1_at_k(pred_spans,               gt_spans, k=K))

        judge = call_judge(
            query=query,
            reference_answer=ref_ans,
            retrieved_context=context,
            system_answer=gen_text,
            model=JUDGE_MODEL,
            base_url=JUDGE_BASE_URL,
            api_key=TRITON_API_KEY,
        )
        if judge.get("failed"):
            judge_failures += 1
        correctness_scores.append(judge["correctness"])
        faithfulness_scores.append(judge["faithfulness"])
        completeness_scores.append(judge["completeness"])

    def mean(xs): return sum(xs) / len(xs) if xs else 0.0

    avg_precision    = mean(precisions)
    avg_recall       = mean(recalls)
    avg_f1           = mean(f1s)
    avg_correctness  = mean(correctness_scores)
    avg_faithfulness = mean(faithfulness_scores)
    avg_completeness = mean(completeness_scores)

    retrieval_score   = (avg_f1 + avg_precision + avg_recall) / 3
    generation_score  = (avg_correctness + avg_faithfulness + avg_completeness / 5) / 3
    leaderboard_score = 0.5 * retrieval_score + 0.5 * generation_score

    return {
        "Total":          {"value": total_queries},
        f"Precision_at{K}": {"value": mean(precisions)},
        f"Recall_at{K}":    {"value": mean(recalls)},
        f"F1_at{K}":        {"value": mean(f1s)},
        "Correctness":    {"value": mean(correctness_scores)},
        "Faithfulness":   {"value": mean(faithfulness_scores)},
        "Completeness":   {"value": mean(completeness_scores)},
        "JudgeFailures":  {"value": judge_failures},
        "LeaderboardScore": {"value": leaderboard_score},
    }


def sample_accumulate_metrics_fn(
    aggregated_metrics: Dict[str, listtype],
) -> Dict[str, Dict[str, Any]]:
    """Weighted average of per-batch metrics across all shards."""
    num_per_batch = [m["value"] for m in aggregated_metrics["Total"]]
    total = sum(num_per_batch)

    weighted_keys = [
        f"Precision_at{K}", f"Recall_at{K}", f"F1_at{K}",
        "Correctness", "Faithfulness", "LeaderboardScore",
    ]
    result = {"Total": {"value": total}}

    for key in weighted_keys:
        result[key] = {
            "value": sum(
                m["value"] * q
                for m, q in zip(aggregated_metrics[key], num_per_batch)
            ) / total,
            "is_algebraic": True,
            "value_range": (0, 1),
        }

    result["Completeness"] = {
        "value": sum(
            m["value"] * q
            for m, q in zip(aggregated_metrics["Completeness"], num_per_batch)
        ) / total,
        "is_algebraic": True,
        "value_range": (0, 5),
    }

    result["JudgeFailures"] = {
        "value": sum(m["value"] for m in aggregated_metrics["JudgeFailures"]),
        "is_algebraic": False,
    }

    return result


### Define Partial Multi-Config Knobs for OpenAI Generator part of RAG Pipeline using RapidFire AI Wrapper APIs

In [ ]:
# Generator model config (shared across all RAG variants)
_GENERATOR_CLIENT_CFG = {
    "api_key":     TRITON_API_KEY,
    "base_url":    "https://tritonai-api.ucsd.edu",
    "max_retries": 2,
}
_GENERATOR_MODEL_CFG = {
    "model":                "api-mistral-small-3.2-2506",
    "max_completion_tokens": 2048,
}

def _make_openai_config(rag_variant: RFLangChainRagSpec) -> RFOpenAIAPIModelConfig:
    return RFOpenAIAPIModelConfig(
        client_config=_GENERATOR_CLIENT_CFG,
        model_config=_GENERATOR_MODEL_CFG,
        rpm_limit=120,
        tpm_limit=1_000_000,
        rag=rag_variant,
        prompt_manager=None,
    )

# One entry per RAG strategy; the grid search will run them in parallel.
# Each entry is labelled so results_df rows are easy to identify.
config_set = List([
    {
        "openai_config":          _make_openai_config(rag_variant),
        "batch_size":             batch_size,
        "preprocess_fn":          openai_sample_preprocess_fn,
        "postprocess_fn":         sample_postprocess_fn,
        "compute_metrics_fn":     sample_compute_metrics_fn,
        "accumulate_metrics_fn":  sample_accumulate_metrics_fn,
        "online_strategy_kwargs": {
            "strategy_name":    "normal",
            "confidence_level": 0.95,
            "use_fpc":          True,
        },
        # Label surfaces in results_df so you can tell the strategies apart
        "config_label": label,
    }
    for rag_variant, label in all_splitters
])


### Create Config Group

In [ ]:
# Simple grid search across all sets of config knob values = 4 combinations in total
config_group = RFGridSearch(config_set)

In [ ]:
# from torch.utils.tensorboard import SummaryWriter

# tensorboard_log_dir = f"/home/yup031/rapidfireai/rapidfire_experiments/{my_experiment}/tensorboard_logs"
# writer = SummaryWriter(log_dir=tensorboard_log_dir)
# writer.add_scalar("test", 0.0, 0)
# writer.close()

# %tensorboard --logdir {tensorboard_log_dir}

### Run Multi-Config Evals

In [ ]:
# Launch evals of all RAG configs in the config_group with swap granularity of 4 chunks
results = experiment.run_evals(
    config_group=config_group,
    dataset=queries,
    num_shards=4,
    seed=42,
)


=== Preprocessing RAG Sources ===


RAG Source ID,Status,Duration,Device,Vector Store
1,Complete,29.9s,CPU,FAISS
2,Complete,29.9s,CPU,FAISS
3,Complete,29.9s,CPU,FAISS



=== Multi-Config Experiment Progress ===


Run ID,Model,Status,Progress,Conf. Interval,text_splitter,chunk_size,chunk_overlap,embedding_cfg.base_url,embedding_cfg.check_embedding_ctx_length,embedding_cfg.class,embedding_cfg.model,vector_store_cfg.batch_size,vector_store_cfg.type,search_cfg.fetch_k,search_cfg.k,search_cfg.lambda_mult,search_cfg.type,reranker_cfg.class,reranker_cfg.model_name,reranker_cfg.top_n,model_config,Completeness,Correctness,F1_at5,Faithfulness,JudgeFailures,LeaderboardScore,Precision_at5,Recall_at5,Samples Processed,Throughput,Total
1,api-mistral-small-3.2-2506,ONGOING,1/4,0.245,LineNumberSplitter,128,32,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,40,20,0.5,mmr,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,3,max_completion_tokens=2048,"3.42 [2.71, 4.12]","0.5833 [0.3418, 0.8249]","0.5833 [0.3418, 0.8249]","0.9167 [0.7812, 1.0000]",0.0000,"0.6787 [0.4499, 0.9075]","0.4722 [0.2276, 0.7168]","0.8333 [0.6507, 1.0000]",12,0.1/s,12
2,api-mistral-small-3.2-2506,ONGOING,1/4,0.244,LineNumberSplitter,128,32,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,40,20,0.5,mmr,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,5,max_completion_tokens=2048,"3.83 [3.13, 4.54]","0.8333 [0.6507, 1.0000]","0.5741 [0.3318, 0.8164]","1.00 [1.00, 1.00]",0.0000,"0.7561 [0.5457, 0.9665]","0.4458 [0.2023, 0.6894]","0.9167 [0.7812, 1.0000]",12,0.1/s,12
3,api-mistral-small-3.2-2506,ONGOING,1/4,0.245,LineNumberSplitter,128,32,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,-,20,-,similarity,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,3,max_completion_tokens=2048,"3.67 [2.96, 4.37]","0.8333 [0.6507, 1.0000]","0.6417 [0.4067, 0.8766]","1.00 [1.00, 1.00]",0.0000,"0.7685 [0.5619, 0.9752]","0.5278 [0.2832, 0.7724]","0.8750 [0.7130, 1.0000]",12,0.1/s,12
4,api-mistral-small-3.2-2506,ONGOING,1/4,0.244,LineNumberSplitter,128,32,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,-,20,-,similarity,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,5,max_completion_tokens=2048,"4.00 [3.29, 4.71]","0.9167 [0.7812, 1.0000]","0.5968 [0.3565, 0.8372]","1.00 [1.00, 1.00]",0.0000,"0.7897 [0.5901, 0.9894]","0.4667 [0.2222, 0.7111]","0.9583 [0.8604, 1.0000]",12,0.1/s,12
5,api-mistral-small-3.2-2506,ONGOING,1/4,0.239,LineNumberSplitter,128,64,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,40,20,0.5,mmr,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,3,max_completion_tokens=2048,"3.67 [2.96, 4.37]","0.6667 [0.4357, 0.8977]","0.6702 [0.4399, 0.9006]","1.00 [1.00, 1.00]",0.0000,"0.7524 [0.5410, 0.9639]","0.6111 [0.3722, 0.8500]","0.8333 [0.6507, 1.0000]",12,0.1/s,12
6,api-mistral-small-3.2-2506,ONGOING,1/4,0.245,LineNumberSplitter,128,64,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,40,20,0.5,mmr,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,5,max_completion_tokens=2048,"3.83 [3.13, 4.54]","0.8333 [0.6507, 1.0000]","0.6032 [0.3635, 0.8429]","1.00 [1.00, 1.00]",0.0000,"0.7672 [0.5601, 0.9743]","0.4833 [0.2385, 0.7282]","0.9167 [0.7812, 1.0000]",12,0.1/s,12
7,api-mistral-small-3.2-2506,ONGOING,1/4,0.243,LineNumberSplitter,128,64,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,-,20,-,similarity,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,3,max_completion_tokens=2048,"3.08 [2.38, 3.79]","0.7500 [0.5378, 0.9622]","0.6750 [0.4455, 0.9045]","1.00 [1.00, 1.00]",0.0000,"0.7593 [0.5498, 0.9687]","0.5556 [0.3121, 0.7990]","0.9583 [0.8604, 1.0000]",12,0.1/s,12
8,api-mistral-small-3.2-2506,ONGOING,1/4,0.244,LineNumberSplitter,128,64,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,-,20,-,similarity,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,5,max_completion_tokens=2048,"3.92 [3.21, 4.62]","1.00 [1.00, 1.00]","0.5913 [0.3504, 0.8321]","1.00 [1.00, 1.00]",0.0000,"0.7992 [0.6030, 0.9955]","0.4625 [0.2182, 0.7068]","0.9583 [0.8604, 1.0000]",12,0.1/s,12
9,api-mis

### View Results

In [ ]:
# Convert results dict to DataFrame – one row per config, one column per metric
results_df = pd.DataFrame([
    {
        "run_id": run_id,
        **{
            k: (v["value"] if isinstance(v, dict) and "value" in v else v)
            for k, v in metrics_dict.items()
        },
    }
    for run_id, (_, metrics_dict) in results.items()
])

# Reorder columns: retrieval metrics first, then judge metrics
retrieval_cols = [c for c in results_df.columns if any(
    c.startswith(p) for p in ["Precision", "Recall", "F1"])]
judge_cols     = ["Correctness", "Faithfulness", "Completeness", "JudgeFailures","LeaderboardScore"]
other_cols     = [c for c in results_df.columns
                  if c not in retrieval_cols + judge_cols + ["run_id", "Total"]]

results_df = results_df[
    ["run_id", "Total"] + retrieval_cols + judge_cols + other_cols
]
results_df


,run_id,Total,Precision_at5,Recall_at5,F1_at5,Correctness,Faithfulness,Completeness,JudgeFailures,LeaderboardScore,model_name,text_splitter,chunk_size,chunk_overlap,embedding_cfg,vector_store_cfg,search_cfg,reranker_cfg,model_config,Samples Processed,Processing Time,Samples Per Second
0,1,45,0.414444,0.888889,0.538437,0.733333,0.911111,3.311111,0,0.691406,api-mistral-small-3.2-2506,LineNumberSplitter,128,32,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...","{'type': 'faiss', 'batch_size': 32}","{'type': 'mmr', 'k': 20, 'fetch_k': 50, 'lambd...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 2048},45,2636.40 seconds,0.02
1,2,45,0.466667,0.838889,0.569312,0.666667,0.955556,3.222222,0,0.690256,api-mistral-small-3.2-2506,LineNumberSplitter,128,32,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...","{'type': 'faiss', 'batch_size': 32}","{'type': 'mmr', 'k': 20, 'fetch_k': 50, 'lambd...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 2048},45,2569.40 seconds,0.02
2,4,45,0.500000,0.883333,0.606349,0.733333,0.955556,3.311111,0,0.723466,api-mistral-small-3.2-2506,LineNumberSplitter,128,32,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...","{'type': 'faiss', 'batch_size': 32}","{'type': 'similarity', 'k': 20}","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 2048},45,2483.14 seconds,0.02
3,5,12,0.316667,0.916667,0.454365,0.666667,1.000000,3.250000,0,0.667394,api-mistral-small-3.2-2506,StructuralRSTSplitter,128,0,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...","{'type': 'faiss', 'batch_size': 32}","{'type': 'mmr', 'k': 20, 'fetch_k': 50, 'lambd...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 2048},12,2442.73 seconds,0.00
4,6,12,0.361111,0.791667,0.483333,0.416667,0.916667,2.500000,0,0.578241,api-mistral-small-3.2-2506,StructuralRSTSplitter,128,0,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...","{'type': 'faiss', 'batch_size': 32}","{'type': 'mmr', 'k': 20, 'fetch_k': 50, 'lambd...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 2048},12,2400.31 seconds,0.00
5,7,45,0.328889,0.888889,0.461199,0.844444,0.911111,3.622222,0,0.693163,api-mistral-small-3.2-2506,StructuralRSTSplitter,128,0,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...","{'type': 'faiss', 'batch_size': 32}","{'type': 'similarity', 'k': 20}","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 2048},45,2359.09 seconds,0.02
6,8,45,0.422222,0.861111,0.545714,0.666667,0.933333,3.155556,0,0.676693,api-mistral-small-3.2-2506,StructuralRSTSplitter,128,0,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...","{'type': 'faiss', 'batch_size': 32}","{'type': 'similarity', 'k': 20}","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 2048},45,2308.86 seconds,0.02
7,9,45,0.262222,0.827778,0.383951,0.822222,0.911111,3.800000,0,0.661214,api-mistral-small-3.2-2506,StructuralRSTSplitter,256,0,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...","{'type': 'faiss', 'batch_size': 32}","{'type': 'mmr', 'k': 20, 'fetch_k': 50, 'lambd...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 2048},45,2268.90 seconds,0.02
8,10,45,0.377778,0.816667,0.499683,0.866667,0.955556,3.777778,0,0.711984,api-mistral-small-3.2-2506,StructuralRSTSplitter,256,0,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...","{'type': 'faiss', 'batch_size': 32}","{'type': 'mmr', 'k': 20, 'fetch_k': 50, 'lambd...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 2048},45,2224.29 seconds,0.02
9,11,45,0.284444,0.877778,0.414286,0.822222,0.955556,3.755556,0,0.684233,api-mistral-small-3.2-2506,StructuralRSTSplitter,256,0,"{'class': 'OpenAIEmbeddings', 'model': 'api-tg...","{'type': 'faiss', 'batch_size': 32}","{'type': 'similarity', 'k': 20}","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_completion_tokens': 2048},45,2180.21 seconds,0.02


In [ ]:
import os

log_dir = f"/home/yup031/rapidfireai/logs/{my_experiment}"
os.makedirs(log_dir, exist_ok=True)

output_path = f"{log_dir}/results.csv"

file_exists = os.path.exists(output_path)
results_df.to_csv(output_path, mode="a", header=not file_exists, index=False)
print(f"{'Appended to' if file_exists else 'Created'} {output_path}")

Appended to /home/yup031/rapidfireai/logs/exp_0501_1/results.csv


### End Experiment

In [ ]:
experiment.end()

NameError: name 'experiment' is not defined

### View RapidFire AI Log Files

In [ ]:
# Get the experiment-specific log file
log_file = experiment.get_log_file_path()

print(f"📄 Log File: {log_file}")
print()

if log_file.exists():
    print("=" * 80)
    print(f"Last 30 lines of {log_file.name}:")
    print("=" * 80)
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            print(line.rstrip())
else:
    print(f"❌ Log file not found: {log_file}")

📄 Log File: /home/yup031/rapidfireai/logs/exp_0501_6/rapidfire.log

Last 30 lines of rapidfire.log:
2026-05-01 17:08:55 | QueryProcessingActor-0 | INFO | query_actor.py:215 | [exp_0501_6:QueryProcessingActor-0] Deserializing FAISS index for this actor...
2026-05-01 17:08:55 | QueryProcessingActor-0 | INFO | query_actor.py:230 | [exp_0501_6:QueryProcessingActor-0] Created independent FAISS vector store for this actor
2026-05-01 17:08:55 | QueryProcessingActor-0 | INFO | query_actor.py:260 | [exp_0501_6:QueryProcessingActor-0] Recreated retriever with search_type=mmr
2026-05-01 17:08:55 | sentence_transformers.base.model | INFO | model.py:962 | No modules.json found for cross-encoder/ms-marco-MiniLM-L6-v2, initializing a new CrossEncoder model.
2026-05-01 17:08:56 | QueryProcessingActor-0 | INFO | query_actor.py:277 | [exp_0501_6:QueryProcessingActor-0] Recreated RAG spec with retrieval mode
2026-05-01 17:09:28 | QueryProcessingActor-0 | INFO | query_actor.py:169 | [exp_0501_6:QueryProce